In [ ]:
# Fix for widget rendering issues
!pip install -q ipywidgets
import ipywidgets as widgets
from IPython.display import display

print("ipywidgets installed and ready")

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
RAW_ROOT = DRIVE_ROOT / "reddit_data" / "raw_yearly"
PROC_ROOT = DRIVE_ROOT / "reddit_data" / "processed"

MONTHLY_ROOT = PROC_ROOT / "monthly"
RESULTS_ROOT = PROC_ROOT / "results"

MONTHLY_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("RAW_ROOT:", RAW_ROOT)
print("PROC_ROOT:", PROC_ROOT)
print("RAW_ROOT exists:", RAW_ROOT.exists())

RAW_ROOT: /content/drive/MyDrive/reddit_data/raw_yearly
PROC_ROOT: /content/drive/MyDrive/reddit_data/processed
RAW_ROOT exists: True


In [ ]:
TARGET_SUBREDDITS = [
    "Decadeology",
    "GenAlpha",
    "GenZ",
    "Youtube",
]

In [ ]:
import os

available_folders = sorted(os.listdir(RAW_ROOT))
print("Available folders in raw_yearly:")
for folder in available_folders:
    print("-", folder)

missing = [s for s in TARGET_SUBREDDITS if s not in available_folders]
print("\nMissing from Drive:", missing if missing else "None")

Available folders in raw_yearly:
- Decadeology
- GenAlpha
- GenZ
- Youtube

Missing from Drive: None


In [ ]:
import json

def stream_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                # skip malformed lines
                continue

In [ ]:
sample_sub = "Youtube"
sample_files = sorted((RAW_ROOT / sample_sub).glob("*.jsonl"))

print(f"{sample_sub} file count:", len(sample_files))
print("First 5 files:")
for p in sample_files[:5]:
    print("-", p.name)

Youtube file count: 19
First 5 files:
- r_youtube_comments_2008.jsonl
- r_youtube_comments_2009.jsonl
- r_youtube_comments_2010.jsonl
- r_youtube_comments_2011.jsonl
- r_youtube_comments_2012.jsonl


In [ ]:
sample_file = sample_files[0]

for i, obj in enumerate(stream_jsonl(sample_file)):
    print("Keys:", obj.keys())
    print("Sample body:", str(obj.get("body", ""))[:300])
    print("Sample created_utc:", obj.get("created_utc"))
    break

Keys: dict_keys(['archived', 'author', 'author_flair_css_class', 'author_flair_text', 'body', 'controversiality', 'created_utc', 'distinguished', 'downs', 'edited', 'gilded', 'id', 'link_id', 'name', 'parent_id', 'retrieved_on', 'score', 'score_hidden', 'subreddit', 'subreddit_id', 'ups', 'permalink'])
Sample body: [deleted]
Sample created_utc: 1201828801


In [ ]:
from datetime import datetime, UTC

def utc_to_month(created_utc):
    try:
        return datetime.fromtimestamp(int(created_utc), UTC).strftime("%Y-%m")
    except Exception:
        return None

def is_valid_comment_text(text):
    if not isinstance(text, str):
        return False
    text = text.strip()
    if not text:
        return False
    if text in {"[deleted]", "[removed]"}:
        return False
    return True

In [ ]:
def build_monthly_files(subreddit_name, raw_root=RAW_ROOT, monthly_root=MONTHLY_ROOT, overwrite=False):
    src_dir = raw_root / subreddit_name
    out_dir = monthly_root / subreddit_name
    out_dir.mkdir(parents=True, exist_ok=True)

    done_file = out_dir / "_DONE.txt"
    summary_file = out_dir / "_SUMMARY.json"

    if done_file.exists() and not overwrite:
        print(f"[{subreddit_name}] Skipping, already processed.")
        return

    if overwrite:
        for old_file in out_dir.glob("*.jsonl"):
            old_file.unlink()
        for meta_file in [done_file, summary_file]:
            if meta_file.exists():
                meta_file.unlink()

    file_count = 0
    kept_count = 0
    skipped_count = 0
    month_counts = {}

    raw_files = sorted(src_dir.glob("*.jsonl"))
    print(f"[{subreddit_name}] Found {len(raw_files)} raw files")

    for file_path in raw_files:
        file_count += 1
        print(f"[{subreddit_name}] Processing {file_path.name}")

        for obj in stream_jsonl(file_path):
            text = obj.get("body", "")
            created_utc = obj.get("created_utc")

            if not is_valid_comment_text(text):
                skipped_count += 1
                continue

            month = utc_to_month(created_utc)
            if month is None:
                skipped_count += 1
                continue

            out_path = out_dir / f"{month}.jsonl"
            out_path.parent.mkdir(parents=True, exist_ok=True)

            record = {
                "text": text,
                "created_utc": created_utc,
                "month": month,
                "subreddit": subreddit_name
            }

            with open(out_path, "a", encoding="utf-8") as wf:
                wf.write(json.dumps(record, ensure_ascii=False) + "\n")

            kept_count += 1
            month_counts[month] = month_counts.get(month, 0) + 1

    summary = {
        "subreddit": subreddit_name,
        "raw_files_processed": file_count,
        "records_kept": kept_count,
        "records_skipped": skipped_count,
        "num_month_files": len(month_counts),
        "first_month": min(month_counts.keys()) if month_counts else None,
        "last_month": max(month_counts.keys()) if month_counts else None,
    }

    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    done_file.write_text("processed", encoding="utf-8")
    print(f"[{subreddit_name}] Finished.")
    print(summary)

In [ ]:
build_monthly_files("GenZ", overwrite=True)

[GenZ] Found 14 raw files
[GenZ] Processing r_GenZ_comments_2013.jsonl
[GenZ] Processing r_GenZ_comments_2014.jsonl
[GenZ] Processing r_GenZ_comments_2015.jsonl
[GenZ] Processing r_GenZ_comments_2016.jsonl
[GenZ] Processing r_GenZ_comments_2017.jsonl
[GenZ] Processing r_GenZ_comments_2018.jsonl
[GenZ] Processing r_GenZ_comments_2019.jsonl
[GenZ] Processing r_GenZ_comments_2020.jsonl
[GenZ] Processing r_GenZ_comments_2021.jsonl
[GenZ] Processing r_GenZ_comments_2022.jsonl
[GenZ] Processing r_GenZ_comments_2023.jsonl
[GenZ] Processing r_GenZ_comments_2024.jsonl
[GenZ] Processing r_GenZ_comments_2025.jsonl
[GenZ] Processing r_GenZ_comments_2026.jsonl
[GenZ] Finished.
{'subreddit': 'GenZ', 'raw_files_processed': 14, 'records_kept': 7631671, 'records_skipped': 371764, 'num_month_files': 104, 'first_month': '2016-01', 'last_month': '2026-03'}


In [7]:
import json

def stream_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue

In [8]:
test_month_file = MONTHLY_ROOT / "GenZ" / "2023-07.jsonl"

print("Exists:", test_month_file.exists())
print("Path:", test_month_file)

if test_month_file.exists():
    for i, obj in enumerate(stream_jsonl(test_month_file)):
        print(obj)
        if i == 1:
            break

Exists: True
Path: /content/drive/MyDrive/reddit_data/processed/monthly/GenZ/2023-07.jsonl
{'text': 'Catholic', 'created_utc': 1688169619, 'month': '2023-07', 'subreddit': 'GenZ'}
{'text': '1999 and 2003 shouldnt be that different, though I wouldnt say they are very similiar.', 'created_utc': 1688169623, 'month': '2023-07', 'subreddit': 'GenZ'}


In [3]:
for sub in ["GenZ", "Youtube", "GenAlpha", "Decadeology"]:
    src_dir = RAW_ROOT / sub
    print(f"\n{sub}")
    print("src exists:", src_dir.exists())
    files = sorted(src_dir.glob("*.jsonl")) if src_dir.exists() else []
    print("raw file count:", len(files))
    print("first 3 files:", [p.name for p in files[:3]])


GenZ
src exists: True
raw file count: 14
first 3 files: ['r_GenZ_comments_2013.jsonl', 'r_GenZ_comments_2014.jsonl', 'r_GenZ_comments_2015.jsonl']

Youtube
src exists: True
raw file count: 19
first 3 files: ['r_youtube_comments_2008.jsonl', 'r_youtube_comments_2009.jsonl', 'r_youtube_comments_2010.jsonl']

GenAlpha
src exists: True
raw file count: 9
first 3 files: ['r_GenAlpha_comments_2018.jsonl', 'r_GenAlpha_comments_2019.jsonl', 'r_GenAlpha_comments_2020.jsonl']

Decadeology
src exists: True
raw file count: 15
first 3 files: ['r_decadeology_comments_2012.jsonl', 'r_decadeology_comments_2013.jsonl', 'r_decadeology_comments_2014.jsonl']
